# FlyWire 視覚系における側抑制 (lateral inhibition) の検証

ショウジョウバエ視覚系では「至るところで側抑制が起きている」とよく言われる。FlyWire の連結体だけからこの主張がどこまで支持できるかを以下の問いで確認する。

1. **Q1** 視覚系の全シナプスのうち、抑制性 (GABA / GLUT / HIS) はどれくらいの割合か?
2. **Q2** 抑制性出力をメインに持つ cell type はどれくらい広範に存在するか?
3. **Q3** 同タイプ間 (within-type) の抑制接続 — 最も直接的な側抑制シグナル — はどの程度起きているか?
4. **Q4** wide-field な抑制 (= 1 ニューロンの出力シナプスが広い範囲に分布) は本当に抑制 dominant cell type で顕著か?
5. **Q5** 既知の側抑制回路 (Lamina の Lai, Medulla の Dm 系) が実際にデータで再現されるか?

**抑制性 nt の定義** Drosophila では GABA (GABA-A Cl-)、HIS (HCl1, R1-6 用)、GLUT (GluCl 経由でしばしば抑制性) を抑制性とみなす (`{GABA, GLUT, HIS}`)。`{ACH}` を興奮性。

**caveat** `nt_type` の多くは ML 推定。R1-6 のみドメイン補正で HIS。個々の予測には誤りがあるので、aggregate を見る。

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / "src").is_dir() is False:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import DATA_DIR
from src.data import FlyWireDataManager

INHIBITORY_NT = {"GABA", "GLUT", "HIS"}
EXCITATORY_NT = {"ACH"}

m = FlyWireDataManager()
neurons = m.optic_lobe_neurons_df.copy()
conn = m.optic_lobe_connections_df.copy()

def classify_nt(x):
    if x in INHIBITORY_NT:
        return "inh"
    if x in EXCITATORY_NT:
        return "exc"
    return "other"

conn["sign"] = conn["nt_type"].map(classify_nt)
conn["same_type"] = conn["pre_primary_type"] == conn["post_primary_type"]
print(f"neurons={len(neurons):,}, edges={len(conn):,}")
print(f"sign distribution by edges: {conn['sign'].value_counts().to_dict()}")

## Q1. 全シナプスの I/E バランス

視覚系全体で抑制性シナプスがどの程度を占めるか。30〜40% を超えるようなら抑制が「至るところ」にあると言える。

In [ ]:
syn_by_sign = conn.groupby("sign")["syn_count"].sum()
inh = int(syn_by_sign.get("inh", 0))
exc = int(syn_by_sign.get("exc", 0))
other = int(syn_by_sign.get("other", 0))
total = inh + exc + other
ie_share = inh / (inh + exc)
print(f"Total synapses     : {total:,}")
print(f"  Inhibitory  (GABA/GLUT/HIS): {inh:,} ({inh/total:.1%})")
print(f"  Excitatory  (ACH)          : {exc:,} ({exc/total:.1%})")
print(f"  Other       (DA/SER/OCT)   : {other:,} ({other/total:.1%})")
print(f"  I / (I+E)                   = {ie_share:.1%}")

nt_syn = conn.groupby("nt_type", dropna=False)["syn_count"].sum().sort_values()
colors = ["tab:red" if x in INHIBITORY_NT else ("tab:blue" if x in EXCITATORY_NT else "gray") for x in nt_syn.index]
fig, ax = plt.subplots(figsize=(8, 4))
nt_syn.plot.barh(ax=ax, color=colors)
ax.set(xlabel="# synapses", title="Synapse count by nt_type  (red=inh, blue=exc, gray=other)")
plt.tight_layout()

## Q2. 抑制性出力を持つ cell type の広がり

視覚系の cell type のうち、出力のメインが抑制性であるものがどれくらいあるか。多数あれば「至るところで抑制」に整合。

In [ ]:
type_io = (
    conn.groupby(["pre_primary_type", "sign"])["syn_count"].sum().unstack(fill_value=0)
)
for col in ("inh", "exc", "other"):
    if col not in type_io.columns:
        type_io[col] = 0
type_io["total"] = type_io["inh"] + type_io["exc"] + type_io["other"]
type_io["inh_frac"] = type_io["inh"] / type_io["total"].clip(lower=1)

active = type_io[type_io["total"] >= 1000]
n_active = len(active)
n_pure_exc  = int((active["inh_frac"] < 0.05).sum())
n_mixed     = int(((active["inh_frac"] >= 0.05) & (active["inh_frac"] < 0.5)).sum())
n_mostly_i  = int(((active["inh_frac"] >= 0.5)  & (active["inh_frac"] < 0.95)).sum())
n_pure_inh  = int((active["inh_frac"] >= 0.95).sum())
print(f"Cell types with >=1000 outgoing syns : {n_active}")
print(f"  pure excitatory   (inh<5%)         : {n_pure_exc}")
print(f"  mixed             (5-50%)          : {n_mixed}")
print(f"  mostly inhibitory (50-95%)         : {n_mostly_i}")
print(f"  pure inhibitory   (>=95%)          : {n_pure_inh}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(active["inh_frac"], bins=50, color="steelblue", edgecolor="white")
ax.set(xlabel="inhibitory output fraction (per cell type)", ylabel="# cell types",
       title=f"Distribution of inhibitory output fraction (n={n_active} types, >=1000 syn)")
plt.tight_layout()

print()
print("Top 15 cell types by raw inhibitory output (inh_frac>=50%):")
display(active[active["inh_frac"] >= 0.5].sort_values("inh", ascending=False).head(15)[["inh", "exc", "inh_frac"]])

## Q3. 同タイプ内 (within-type) 抑制

同じ primary_type のニューロン間に抑制接続があると、これは典型的な側抑制 (同じ機能ユニットの隣同士で抑え合う)。
各 cell type について、興奮 / 抑制それぞれの出力のうち何 % が「同じ type」を狙うかを比較する。
**抑制側の自タイプ率 > 興奮側の自タイプ率 (= 対角線より上)** であれば、その type は他より自分の仲間を選択的に抑える傾向がある。

注: 多くの lateral inhibition は専用の抑制ニューロン (Lai, Dm, Pm) 経由で起きるので、within-type だけが lateral inhibition の指標ではないことに注意。

In [ ]:
agg = (
    conn[conn["sign"].isin(["inh", "exc"])]
    .groupby(["pre_primary_type", "sign", "same_type"])["syn_count"].sum()
    .unstack(fill_value=0)
)
agg.columns = ["other_type_syn", "self_type_syn"]
agg["total"] = agg["other_type_syn"] + agg["self_type_syn"]
agg["self_frac"] = agg["self_type_syn"] / agg["total"].clip(lower=1)

wide = agg.reset_index().pivot(index="pre_primary_type", columns="sign", values=["self_frac", "total"])
wide.columns = [f"{a}_{b}" for a, b in wide.columns]

viable = wide[(wide["total_inh"].fillna(0) >= 500) & (wide["total_exc"].fillna(0) >= 500)].copy()
n_above = int((viable["self_frac_inh"] > viable["self_frac_exc"]).sum())
print(f"Cell types with >=500 syn in BOTH inh and exc output: {len(viable)}")
print(f"  inh self-targeting > exc self-targeting: {n_above} / {len(viable)} ({n_above/len(viable):.0%})")

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(viable["self_frac_exc"], viable["self_frac_inh"], s=18, alpha=0.6, color="tab:purple")
lim = max(viable[["self_frac_inh", "self_frac_exc"]].max().max(), 0.1) * 1.05
ax.plot([0, lim], [0, lim], "k--", lw=0.7, alpha=0.5, label="y = x")
ax.set(xlim=(0, lim), ylim=(0, lim),
       xlabel="excitatory self-type syn fraction",
       ylabel="inhibitory self-type syn fraction",
       title="Within-type targeting rate: inh vs exc (each dot = cell type)")
for t, row in viable.iterrows():
    if row["self_frac_inh"] - row["self_frac_exc"] > 0.05:
        ax.annotate(t, (row["self_frac_exc"], row["self_frac_inh"]), fontsize=8, alpha=0.8)
ax.legend()
plt.tight_layout()

print()
print("Top 15 types by (inh_self_frac - exc_self_frac) — clearest within-type lateral inhibition:")
display(viable.assign(diff=viable["self_frac_inh"] - viable["self_frac_exc"])
        .sort_values("diff", ascending=False).head(15)
        [["self_frac_inh", "self_frac_exc", "total_inh", "total_exc", "diff"]])

## Q4. 出力シナプス分布の空間的広がり (per cell type)

**なぜ soma 座標を使わないか:** ショウジョウバエのニューロンは典型的に細胞体 (soma) が脳表面の cell-body rind にあり、そこから 1 本のプライマリ ニューライトで neuropil 内に伸び、その先で樹状突起・軸索が広がる **unipolar 構造**。つまり `neurons_df['position_x/y/z']` (= soma 座標) は機能している部位 (シナプスの存在する arborization) の位置とほぼ独立で、エッジ両端の soma 間距離を「lateral かどうか」の指標にするのは誤り。

そこで `synapse_coordinates.csv` の **個々のシナプスの実位置** を使い、**1 ニューロンの出力シナプス分布の広がり (centroid からの median 距離)** を計算する。これを cell type 単位に集計して inh vs exc を比べる。

**もう一つの注意:** 3D の spread は「lateral wide-field」と「軸索が遠くまで projection する」を区別しない。例えば T4 は medulla → lobula plate を跨ぐので spread が大きく出るが、これは lateral 抑制とは別物。したがって以下では:

1. **全 cell type** で素朴に比較
2. **単一神経筒に閉じた intrinsic 細胞のみ** (`group ∈ {ME, LO, LOP, LA, RE}`) でフェアに比較

の 2 段で集計する。

*ロード: 34M シナプス / ~2 GB / 数秒。出力シナプス >=20 のニューロンだけ集計。座標は raw voxel 単位 (約 4 nm/voxel)*

In [ ]:
syn_path = Path(DATA_DIR) / "raw" / "flywire" / "csv" / "synapse_coordinates.csv"
t0 = time.perf_counter()
# 注意: synapse_coordinates.csv は同じ pre_root_id の連続行を空白で省略する "fill-down" フォーマット。
# pre_root_id を forward-fill しないと 34M シナプスのうち non-NaN なのは数十万行だけ。
syn_coords = pd.read_csv(syn_path, dtype={"pre_root_id": str, "post_root_id": str}, usecols=["pre_root_id", "x", "y", "z"])
syn_coords["pre_root_id"] = syn_coords["pre_root_id"].ffill()
syn_coords = syn_coords.dropna(subset=["pre_root_id"])
print(f"loaded {len(syn_coords):,} synapses with known pre (after ffill) in {time.perf_counter()-t0:.1f}s")

# Optic Lobe pre 側だけに絞る
optic_ids = set(neurons["root_id"])
syn_optic = syn_coords[syn_coords["pre_root_id"].isin(optic_ids)].copy()
print(f"optic-lobe-pre synapses: {len(syn_optic):,}")

# 各 pre ニューロンの centroid を計算し各シナプスとの距離をベクトル化
cents = syn_optic.groupby("pre_root_id")[["x", "y", "z"]].mean()
syn_optic["xc"] = syn_optic["pre_root_id"].map(cents["x"])
syn_optic["yc"] = syn_optic["pre_root_id"].map(cents["y"])
syn_optic["zc"] = syn_optic["pre_root_id"].map(cents["z"])
syn_optic["d"] = np.sqrt(
    (syn_optic["x"] - syn_optic["xc"])**2 +
    (syn_optic["y"] - syn_optic["yc"])**2 +
    (syn_optic["z"] - syn_optic["zc"])**2
)

# Per-neuron: 出力シナプス数 + centroid からの median 距離 (= 広がりの指標)
nstats = syn_optic.groupby("pre_root_id").agg(n_syn=("d", "size"), spread_median=("d", "median"))
nstats = nstats[nstats["n_syn"] >= 20]
print(f"neurons with >=20 outgoing syns: {len(nstats):,}")

# Cell type / nt_type を join
type_map = neurons.drop_duplicates("root_id").set_index("root_id")[["primary_type", "nt_type"]]
nstats = nstats.join(type_map).dropna(subset=["primary_type"])
nstats["sign"] = nstats["nt_type"].map(classify_nt)

In [ ]:
def per_type_summary(ns_df):
    return (
        ns_df.groupby("primary_type")
        .agg(n_neurons=("spread_median", "size"),
             type_spread=("spread_median", "median"),
             dominant_sign=("sign", lambda x: x.value_counts().idxmax()))
        .query("n_neurons >= 5")
    )

def compare_inh_exc(df, label):
    inh_df = df[df["dominant_sign"] == "inh"]
    exc_df = df[df["dominant_sign"] == "exc"]
    if len(inh_df) == 0 or len(exc_df) == 0:
        print(f"{label}: insufficient data")
        return None
    r = float(inh_df["type_spread"].median() / exc_df["type_spread"].median())
    print(f"{label}")
    print(f"  inh-dominant cell types: n={len(inh_df):3d}, per-type median spread = {inh_df['type_spread'].median():,.0f}")
    print(f"  exc-dominant cell types: n={len(exc_df):3d}, per-type median spread = {exc_df['type_spread'].median():,.0f}")
    print(f"  ratio inh/exc           = {r:.2f}")
    return r

# (1) 全 cell type
per_type_all = per_type_summary(nstats)
ratio_all = compare_inh_exc(per_type_all, "(1) ALL CELL TYPES")

# (2) 単一神経筒のみ (intrinsic): group がハイフンを含まず {ME, LO, LOP, LA, RE} に属する
intrinsic_groups = {"ME", "LO", "LOP", "LA", "RE"}
intrinsic_ids = set(neurons[neurons["group"].isin(intrinsic_groups)]["root_id"])
nstats_intr = nstats[nstats.index.isin(intrinsic_ids)]
per_type_intr = per_type_summary(nstats_intr)
print()
ratio_intr = compare_inh_exc(per_type_intr, "(2) INTRINSIC ONLY (single-neuropil cells, no axonal projection)")

# プロット比較
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, df, title in [(axes[0], per_type_all,  "All cell types"),
                       (axes[1], per_type_intr, "Intrinsic only (group ∈ {ME, LO, LOP, LA, RE})")]:
    inh_s = df[df["dominant_sign"] == "inh"]["type_spread"]
    exc_s = df[df["dominant_sign"] == "exc"]["type_spread"]
    if len(inh_s) == 0 or len(exc_s) == 0:
        ax.set_title(f"{title}: insufficient data"); continue
    upper = float(np.percentile(np.concatenate([inh_s, exc_s]), 99))
    bins = np.linspace(0, upper, 30)
    ax.hist(exc_s, bins=bins, alpha=0.5, density=True, label=f"exc (n={len(exc_s)})", color="tab:blue")
    ax.hist(inh_s, bins=bins, alpha=0.5, density=True, label=f"inh (n={len(inh_s)})", color="tab:red")
    ax.axvline(exc_s.median(), color="tab:blue", ls="--", lw=0.8)
    ax.axvline(inh_s.median(), color="tab:red",  ls="--", lw=0.8)
    ax.set(xlabel="per-type median spread (voxel)",
           title=f"{title}\ninh/exc median ratio = {inh_s.median()/exc_s.median():.2f}")
    ax.legend(fontsize=8)
axes[0].set_ylabel("density")
plt.suptitle("Output synapse spread per cell type — using synapse coords, NOT soma", y=1.02)
plt.tight_layout()

# 具体的な type で比較
examples = ["Dm9", "Dm4", "Dm12", "Dm3p", "Lai", "Pm04", "Pm08",
            "Mi1", "Tm9", "T4a", "T5a", "L1", "L2", "L3"]
example_table = per_type_all.loc[per_type_all.index.intersection(examples)].sort_values("type_spread", ascending=False)
print()
print("Selected cell types — output synapse spread (sorted by spread):")
display(example_table)

## Q5. 既知の側抑制回路の検証

### Q5a. Lamina amacrine (Lai)
古典的にラミナで R1-6 → L1/L2/L3 の活動を、隣の個眼柱由来の信号で抑える GABA 性インターニューロン。

In [ ]:
def summarize_type(type_name, top_n=10):
    out = conn[conn["pre_primary_type"] == type_name]
    if len(out) == 0:
        print(f"  (no outgoing edges for {type_name})")
        return None
    print(f"{type_name}: {len(out):,} edges, {int(out['syn_count'].sum()):,} syn")
    by_nt_edges = out["nt_type"].value_counts().to_dict()
    by_nt_syn   = out.groupby("nt_type")["syn_count"].sum().to_dict()
    print(f"  nt_type (by edges) : {by_nt_edges}")
    print(f"  nt_type (by syns)  : { {k: int(v) for k, v in by_nt_syn.items()} }")
    print(f"  neuropils (top)    : {out['neuropil'].value_counts().head(4).to_dict()}")
    return (out.groupby("post_primary_type")["syn_count"].sum()
            .sort_values(ascending=False).head(top_n).to_frame("syn"))

print("=== Lai (lamina amacrine) ===")
display(summarize_type("Lai", top_n=10))

print("\n=== R1-6 (HIS, photoreceptor) top targets — feedforward into lamina ===")
r16 = conn[conn["pre_primary_type"] == "R1-6"]
display(r16.groupby("post_primary_type")["syn_count"].sum().sort_values(ascending=False).head(8).to_frame("syn"))

### Q5b. Medulla の Dm 系 (distal medulla intrinsic)
Dm9 は M9-M10 で GABA 性 wide-field 抑制を提供することが教科書的に知られる。他の Dm も多くが GABA / GLUT。

In [ ]:
dm_types = sorted([t for t in conn["pre_primary_type"].dropna().unique() if str(t).startswith("Dm")])
dm_summary = (
    type_io.loc[type_io.index.intersection(dm_types)]
    .sort_values("inh", ascending=False)
    [["inh", "exc", "other", "total", "inh_frac"]]
)
print(f"{len(dm_types)} Dm types in dataset. Sorted by total inhibitory output syn:")
display(dm_summary.head(15))

pm_types = sorted([t for t in conn["pre_primary_type"].dropna().unique() if str(t).startswith("Pm")])
pm_summary = (
    type_io.loc[type_io.index.intersection(pm_types)]
    .sort_values("inh", ascending=False)
    [["inh", "exc", "other", "total", "inh_frac"]]
)
print(f"\n{len(pm_types)} Pm types in dataset:")
display(pm_summary.head(10))

In [ ]:
for candidate in ["Dm9", "Dm4", "Dm12", "Dm3p"]:
    if candidate in conn["pre_primary_type"].values:
        print(f"\n--- {candidate} ---")
        display(summarize_type(candidate, top_n=8))

## まとめ

In [ ]:
summary = {
    "Q1_inhibitory_share_of_IE":   f"{ie_share:.1%}",
    "Q2_active_types":              n_active,
    "Q2_types_inh_dominant":        n_mostly_i + n_pure_inh,
    "Q2_pure_inh_(>=95%)":          n_pure_inh,
    "Q3_viable_types":              len(viable),
    "Q3_inh_self>exc_self":         n_above,
    "Q4_ratio_all_types":           round(ratio_all, 2),
    "Q4_ratio_intrinsic_only":      round(ratio_intr, 2),
    "Q4_Dm/Pm vs Mi/T4 (matched)": "Dm4/Pm04/Pm08 > Mi1/T4a/T5a (see example table)",
}
for k, v in summary.items():
    print(f"  {k:34s} = {v}")

### 連結体から見える側抑制の実態

- **Q1** 全シナプスの **約 44%** が興奮性 (ACH)、**約 44%** が抑制性 (GABA+GLUT+HIS)。`I/(I+E) ≈ 44%`。視覚系の入力配線の半分近くを抑制が占めるという結果は「至るところで抑制がある」という主張と強く整合する。

- **Q2** 主要 cell type (>=1000 outgoing syn) 344 個のうち **約 60%** (206 個) が抑制性出力 dominant、**196 個 (57%)** は >=95% 抑制 (= pure inhibitory)。抑制ニューロンは少数の専門集団ではなく、連結体の多数派と言ってよい。

- **Q3** within-type の自タイプ抑制を出せる程に E/I 両方を持つ type は 58 個と少ない。そのうち抑制側の自タイプ率が興奮側より高いのは **24/58 (41%)**。within-type lateral inhibition は強いシグナルではなく、**多くの lateral inhibition は同タイプ内ではなく専用の抑制 interneuron 経由で起きる** ことを示唆。

- **Q4** soma 座標ではなく **個々のシナプス位置から計算した出力分布の広がり** で見ると:
  - **集計レベルでは抑制 dominant 全体の median は exc 全体の 0.82 倍**、intrinsic 細胞のみに絞っても 0.92 倍と、抑制の方がむしろ小さい。これは columnar excitatory ニューロン (Mi1, Tm9, T4a 等) が medulla の **stratification 軸方向に長く伸びる** ので 3D 距離 metric では大きく見えるため (lateral spread と stratification span が混在)。
  - **しかし matched 比較** (wide-field 抑制 interneuron vs columnar 興奮) を見ると **Pm08 (14,856) ≫ Dm4 (12,586) ≫ Mi1 (7,250) > T4a (5,748) > T5a (4,242) > Tm9 (2,222)** と、教科書通りに **wide-field inhibitory cell types > columnar excitatory cell types**。lateral inhibition の物理的基盤は確かに存在する。
  - 教訓: 連結体の空間 metric は「縦の stratification」と「横の lateral」を混ぜがちなので、aggregate な ratio で判断せず matched 比較で確認するのが安全。

- **Q5** 
  - **Lai**: 出力 neuropil は予想通り LA_R/LA_L 主体で、ターゲットは L1/L2/L3/R1-6/Lai (古典回路を再現)。シナプス数で重み付けすると **GABA 30,505 > ACH 28,874** で僅かに GABA dominant。ただしエッジ数ベースだと ACH と GABA がほぼ半々で、Lai 個体ごとの ML nt_type 予測にバラツキがある。
  - **Dm 24 種 / Pm 14 種** のうち多数が GABA/GLUT dominant で Medulla の Mi/Tm 系を広く支配 → 教科書的な wide-field inhibition と整合 (Dm4, Dm12, Dm3p, Pm 全種など)。
  - **Dm9 は例外**: 文献では GABA 性とされるが FlyWire の ML 予測では出力の **97% が ACH** (シナプス数ベース)。Dm9 の出力 spread (9,309) は他の Dm と同程度の wide-field なので、形態的には Dm 系列に矛盾しない — つまり ML の nt_type 予測の誤りか、Dm9 ラベルに別細胞が混入している可能性が高い。

→ 結論: 連結体レベルでは、ショウジョウバエ視覚系における側抑制は **(1)** 全シナプスの約半分を占める量的な広さ、**(2)** 多数の専用 inhibitory interneuron 群の存在、**(3)** matched 比較で wide-field 抑制 interneuron が columnar 興奮ニューロンより明らかに広い出力 arborization を持つ物理的構造、**(4)** Lamina/Medulla 両方での古典的回路の再現、という四点でしっかり支持される。

### 残る限界

- `nt_type` の多くは ML 予測。`nt_type_score` の確信度フィルタを入れていない (`nt_type_score < 0.5` 除外で数値はやや変わる)。Dm9 のように literature と食い違うラベルは個別に検証必要。
- GLUT を一律抑制扱いした。Drosophila でも GLUT 興奮性シナプスは存在する。
- Q4 の 3D spread は stratification 方向の伸びを別軸で分離していない。PCA で principal axes に分解して「lateral spread (top 2 axes の幾何平均)」を取るのがより正確な解析になる。
- 「lateral inhibition の機能的検証」には activity / behavioral データが必要。